In [ ]:
# This is injectable via the parameters flag of papermill
csv_filepath = "womens-clothing-ecommerce.csv"

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import pandas as pd
from nemo_platform import NeMoPlatform
from nemo_platform.beta.safe_synthesizer.job_builder import SafeSynthesizerJobBuilder

base_url = os.environ.get("NMP_BASE_URL") or "http://localhost:8080"

client = NeMoPlatform(
    base_url=base_url,
)
data = pd.read_csv(csv_filepath)  # noqa
builder = SafeSynthesizerJobBuilder(client).with_data_source(data)

# Configuration flags
skip_synthesis = os.environ.get("SKIP_SYNTHESIS", "").lower() in ["true", "1", "yes", "y"]
skip_pii_replacement = os.environ.get("SKIP_PII_REPLACEMENT", "").lower() in ["true", "1", "yes", "y"]
transform_only = os.environ.get("TRANSFORM_ONLY", "").lower() in ["true", "1", "yes", "y"]

expected_shape = (1000, 11)

if transform_only:
    # Transform only mode: Skip synthesis, do PII replacement without GLiNER
    print("🔧 Running in TRANSFORM_ONLY mode (no synthesis, no GLiNER)")
    builder.with_replace_pii(**{"globals": {"ner": {"gliner": {"enable_gliner": False}}}})
    expected_shape = (1001, 11)
else:
    # Standard mode: Use SKIP_SYNTHESIS and SKIP_PII_REPLACEMENT flags
    if not skip_synthesis:
        builder.synthesize()
    if not skip_pii_replacement:
        builder.with_replace_pii()
        if skip_synthesis:
            expected_shape = (1001, 11)

In [ ]:
import time

# Conditionally create HF token secret if HF_TOKEN is set in environment
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    secret_name = f"hf-token-nss-test-{int(time.time())}"
    client.secrets.create(workspace="default", name=secret_name, value=hf_token)
    print(f"✓ Created HF token secret: {secret_name}")
    builder = builder.with_hf_token_secret(secret_name)

# Conditionally create model provider for column classification if NIM_API_KEY is set
api_key = os.environ.get("NIM_API_KEY")
if api_key:
    timestamp = int(time.time())
    # Create the API key as a secret first
    api_key_secret_name = f"nim-api-key-nss-test-{timestamp}"
    client.secrets.create(workspace="default", name=api_key_secret_name, value=api_key)
    print(f"✓ Created API key secret: {api_key_secret_name}")

    # Create the model provider referencing the secret
    provider_name = f"classify-llm-nss-test-{timestamp}"
    client.inference.providers.create(
        workspace="default",
        name=provider_name,
        host_url="https://integrate.api.nvidia.com/v1",
        api_key_secret_name=api_key_secret_name,
        description="Model provider for Safe Synthesizer column classification",
    )
    print(f"✓ Created model provider: {provider_name}")
    builder = builder.with_classify_model_provider(provider_name)

In [ ]:
job_name = f"synthesis-test-{pd.Timestamp.now().strftime('%Y%m%d-%H%M%S')}"
project_name = "test-project"

try:
    client.projects.create(workspace="default", name=project_name)
except Exception:
    pass

job = builder.create_job(name=job_name, project=project_name)
job.wait_for_completion()

In [ ]:
print(job.fetch_status_info())
current_status = job.fetch_status()
expected_status = "completed"

assert current_status == expected_status, f"Job status mismatch. Expected: '{expected_status}', Got: '{current_status}'"

In [ ]:
if not skip_synthesis and not transform_only:
    job.display_report_in_notebook()

In [ ]:
current_shape = job.fetch_data().shape

assert expected_shape == current_shape, f"Result shape mismatch. Expected: '{expected_shape}', Got: '{current_shape}'"

In [ ]:
# TODO: enable once filter is working
# jobs_response = client.safe_synthesizer.jobs.list(workspace="default", extra_query={"filter[name]": ["synthesis"]})
# print(f"Found {len(jobs_response.data)} jobs matching 'synthesis'")
# assert len(jobs_response.data) >= 1, "Should find at least our job"
#
# matching_job = next((j for j in jobs_response.data if j.name == job_name), None)
# assert matching_job is not None, f"Should find our job {job_name}"
# print(f"✓ Found job by name filter: {matching_job.name}")

In [ ]:
# project_jobs = client.safe_synthesizer.jobs.list(workspace="default", extra_query={"filter[project]": ["test-project"]})
# print(f"Found {len(project_jobs.data)} jobs in project matching 'test-project'")
# assert len(project_jobs.data) >= 1, "Should find at least our job"
#
# matching_project_job = next((j for j in project_jobs.data if j.project == project_name), None)
# assert matching_project_job is not None, f"Should find our job in project {project_name}"
# print(f"✓ Found job by project filter: {matching_project_job.name} in {matching_project_job.project}")

In [ ]:
# combined_filter = client.safe_synthesizer.jobs.list(
#    workspace="default", extra_query={"filter[name]": ["synthesis"], "filter[project]": ["test"]}
# )
# print(f"Found {len(combined_filter.data)} jobs matching both 'synthesis' AND 'test'")
# assert len(combined_filter.data) >= 1, "Should find our job with combined filter"
#
# matching_combined = next((j for j in combined_filter.data if j.id == job.job_id), None)
# assert matching_combined is not None, "Should find our job with combined filter"
# print(f"✓ Found job by combined filter: {matching_combined.name}")